# CholeGuard — entraînement sur GPU Kaggle
Segmentation (Endoscapes + CholecSeg8k), têtes CVS (ablation image / sans masques / guidée par l'anatomie), évaluation, export ONNX, publication des artefacts sur Hugging Face.

Prérequis : dataset `newslab/cholecseg8k` ajouté en *Input*, *Accelerator* = GPU, *Internet* = activé, secret Kaggle `HF_TOKEN` (jeton Hugging Face en écriture).

In [ ]:
!git clone -q -b develop https://github.com/azddine123/CholeGuard.git /kaggle/working/CholeGuard
%cd /kaggle/working/CholeGuard
!pip install -q segmentation-models-pytorch timm albumentations onnx onnxruntime kagglehub
!pip install -q -e .

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!bash -o pipefail -c 'bash scripts/kaggle_pipeline.sh 2>&1 | grep -v Warning'

In [ ]:
import json; m = json.load(open('/kaggle/working/out/metrics.json'))
print(json.dumps({k: {x: m['cvs'][k][x] for x in ('mean_ap', 'mean_bacc')} for k in m['cvs']}, indent=1))
print(json.dumps(m['signal']['test'], indent=1)); print(m['segmentation']['test_dice_fg'])

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
api = HfApi(token=UserSecretsClient().get_secret('HF_TOKEN'))
repo = api.whoami()['name'] + '/choleguard'
api.create_repo(repo, exist_ok=True)
api.upload_folder(folder_path='/kaggle/working/out', repo_id=repo, allow_patterns=['*.onnx', '*.pt', '*.json', '*.yaml', 'figures/*'])
print('https://huggingface.co/' + repo)